In [ ]:
# import 

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys

#activate interactive figures
#%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

In [ ]:
## Read all data
%autoreload 2
from analyses_level2.read_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
#print(all_data)
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 #'O3'
                 ] # define data to read in. If empty, all data is used 

## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : False # exclude flagged flask-data
}

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(species=sel)
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="species")


In [ ]:
## check flask data
sel_spec = 'CO2_flask'

ds = ds_all.sel(species=sel_spec)
ds = ds.where(~np.isnan(ds.value), drop=True) # remove times where we have no data
ds.QCflag.plot(ls='',marker='.')


In [ ]:
#select all flask species: 
sel_species = [s for s in selected_data if 'flask' in s]

fig, axs = plt.subplots(len(sel_species),1,sharex=True)
for s,ax in zip(sel_species,axs):
    ds = ds_all.sel(species=s)
    ds.value.plot(ax=ax,ls='',marker='.')
    
    # check QCflag = 3
    ds.where(ds.QCflag==3).value.plot(ax=ax,ls='',marker='.',c='r')

    #check original flag (reject if first character is not '.' ) -> gives the same!
    #mask_flag = [str(s)[0] != '.' if isinstance(s, str) else False for s in ds.ORG_QCflag.values]
    #mask_dataarray = xr.DataArray(mask_flag, dims='time', coords={'time': ds['time']})
    #if any(mask_flag):
    #    ds.where(mask_dataarray,drop=True).value.plot(ax=ax,ls='',marker='x',c='g')


#### check ozone data

In [ ]:
import nappy
data_path = '../data/wdc/ebas/air/'
for file in os.listdir(data_path):
    if "ozone" in str(file):
       print(file)

file


In [ ]:
%autoreload 2
from processing import ebas
df = ebas.compile_ebas_ozone_data_into_dataframe(data_path,read_unc=True)

In [ ]:
df.index

In [ ]:
pd.to_datetime(df.index[0])#.astype('DateTime64DType')
df.index[0]

In [ ]:
pd.to_datetime(df.index)

In [ ]:
df.to_xarray().O3.plot(marker='o',alpha=0.2,ls='')

In [ ]:
## wdc data: 
from analyses_level2.read_data import AvailableData, create_data_reader

data_path = "../data/"
sel='CO2'
data_reader =  create_data_reader(data_path,sel) #creates an instance of the desired data_reader class
print(f"Data from {data_reader.__class__.__name__} for {sel}:")

# call the data-reading function on that instance: 
data = data_reader.read_data() 

In [ ]:
data.index

In [ ]:
data.index[0]